# Fine-tune Gemma 4 E2B as a model router

This Colab QLoRA-tunes **Gemma 4 E2B Instruct** with **Unsloth** so it routes a software task to one model:

- `luna` = simple / bounded change
- `astra` = complex / security-sensitive / architectural work

It starts from the `ModernBERT_train_classify.ipynb` tasks and adds harder ones. ModernBERT learns a classification head. This notebook teaches Gemma to answer with the route name. The prompt only names the two routes. The labels are what carry the policy.

A free T4 is enough for 4-bit QLoRA. In Colab use **Runtime → Change runtime type → T4 GPU**.

Gemma weights are gated. Accept the license on [google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it) and sign in if the download returns 401.


In [ ]:
!nvidia-smi


## 1. Install the Gemma 4 / Unsloth stack

Same install as the Gemma 4 E2B tweet notebook, so the chat template and 4-bit loader match current Unsloth Gemma 4 notebooks.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -U unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10':'0.0.34',
        '2.9':'0.0.33.post1',
        '2.8':'0.0.32.post2'
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install torchcodec
!pip install --no-deps --upgrade timm

import torch
torch._dynamo.config.recompile_limit = 64


## 2. Configuration


In [ ]:
# ---------- Model ----------
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

# ---------- Training ----------
LORA_R = 8
LORA_ALPHA = 8
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4
WARMUP_STEPS = 2
SEED = 3407

# Stratified split over the expanded labeled set. The original 32 tasks
# are included, but this is no longer the ModernBERT notebook's split.
SPLIT_SEED = 42
TEST_SIZE = 0.25

# Do not define luna vs astra here. The base model was matching the
# fine-tune at 100% when this prompt listed the routing rules.
ROUTER_SYSTEM_PROMPT = (
    "You are a model router for software tasks. "
    "Reply with only one route name: luna or astra."
)


## 3. Dataset

`0` is `luna`, `1` is `astra`. The first 32 tasks are the ModernBERT demo. The rest add more of those cases, plus hard negatives: security or migration wording that is still a bounded change, and ordinary-sounding work that changes persistence, auth, or service boundaries.

The split is stratified with `random_state=42`. The paraphrase list stays out of that split. Those lines are the check for whether fine-tuning beat the base model.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

ID2ROUTE = {0: "luna", 1: "astra"}

# The first 32 tasks are the ModernBERT demo. Later rows are extra
# examples, including hard negatives.
examples = [
    # Luna: simple / bounded
    ("Fix a typo in the README.", 0),
    ("Change the primary button color from blue to green.", 0),
    ("Rename a local variable for clarity.", 0),
    ("Add a missing unit test for an existing helper.", 0),
    ("Update an API error message.", 0),
    ("Add pagination to an existing endpoint.", 0),
    ("Refactor duplicate validation logic into a shared helper.", 0),
    ("Add structured logging around failed HTTP requests.", 0),
    ("Update the OAuth login page copy without changing authentication logic.", 0),
    ("Rename the authentication middleware function without changing behavior.", 0),
    ("Add a runtime-only field to the User object that is not persisted.", 0),
    ("Add another field to an API response using data already in memory.", 0),
    ("Improve the wording of a validation error.", 0),
    ("Add a new test case for an existing endpoint.", 0),
    ("Document how to run the development server.", 0),
    ("Extract a small helper function without changing behavior.", 0),

    # Astra: complex / risky
    ("Implement Google OAuth login and persist users in PostgreSQL.", 1),
    ("Add refresh token rotation and revocation.", 1),
    ("Migrate user IDs from integers to UUIDs.", 1),
    ("Replace session authentication with JWT authentication.", 1),
    ("Add role-based access control for admins and normal users.", 1),
    ("Create an audit log table and record security-sensitive user actions.", 1),
    ("Add an encrypted API key field to the User model and persist it.", 1),
    ("Redesign the authorization layer across multiple services.", 1),
    ("Implement password reset using signed expiring tokens.", 1),
    ("Add multi-tenant permissions and migrate existing authorization data.", 1),
    ("Change token issuance rules and refresh-token storage.", 1),
    ("Move authentication state from database-backed sessions to stateless tokens.", 1),
    ("Add a new persisted field to User and write the required database migration.", 1),
    ("Introduce an event-driven workflow across the API and worker services.", 1),
    ("Split the monolith authentication module into separate services.", 1),
    ("Add end-to-end encryption for stored customer credentials.", 1),

    # More bounded work, including wording that mentions auth or data.
    ("Fix the broken link in the contributing guide.", 0),
    ("Change the font size of the settings page heading.", 0),
    ("Rename the loop index in the invoice formatter.", 0),
    ("Add a test that the existing slug helper rejects an empty string.", 0),
    ("Update the empty-state copy on the dashboard.", 0),
    ("Add a page-size query param to an endpoint that already returns a list in memory.", 0),
    ("Deduplicate two identical date-parsing snippets into one helper with the same result.", 0),
    ("Log the request id when an existing HTTP client retries.", 0),
    ("Change the placeholder text in the email field.", 0),
    ("Rename authToken to sessionToken in one function without changing what is read or written.", 0),
    ("Add a transient displayName on the User view model. It is not stored.", 0),
    ("Include the already-loaded team name in the profile response.", 0),
    ("Soften the wording of the file-too-large message.", 0),
    ("Add one assertion to the billing calculator test. The calculator stays the same.", 0),
    ("Write down the command that starts the worker process.", 0),
    ("Pull the color contrast check into a helper. Output stays the same.", 0),
    ("Fix the typo in the password-reset email subject.", 0),
    ("Add a docstring to the AES wrapper. Do not change the key size or mode.", 0),
    ("Add a screenshot of the current admin screen to the handbook.", 0),
    ("Change the login button label from Sign in to Log in.", 0),
    ("Sort the in-memory notification list before returning it. Do not touch the table.", 0),
    ("Correct the comment above the migration that created the users table.", 0),
    ("Add a unit test that stubs the existing permission check and expects the current allow result.", 0),
    ("Update the OpenAPI description of /users without changing the handler.", 0),
    ("Replace the blue border on the error toast with a gray border.", 0),
    ("Rename the local jwt variable in a test to token.", 0),
    ("Add a console log when the development server finishes compiling.", 0),
    ("Explain in the README which environment variables the app already reads.", 0),
    ("Reformat the existing user seed SQL so it is readable. Keep the same rows.", 0),
    ("Add a loading state to the save button. The request stays the same.", 0),
    ("Fix a spelling mistake in the RBAC section of the internal wiki.", 0),
    ("Extract the Bearer prefix into a constant used by the current client.", 0),
    ("Add a storybook example for the existing avatar component.", 0),
    ("Update the copyright year in the footer.", 0),
    ("Make the sidebar collapse animation 50ms faster.", 0),
    ("Add a regression test that a mocked failed HTTP request still logs the same message.", 0),
    ("Clarify the validation error that says bad input so it says name is required.", 0),
    ("Move date-formatting helpers from db.py into dates.py. No query changes.", 0),
    ("Add a tooltip on the existing disabled deploy button.", 0),
    ("Document that session data currently lives in Postgres, without changing where it lives.", 0),

    # More risky work, including requests that sound small.
    ("Add sign-in with GitHub and store the linked account in the database.", 1),
    ("Expire refresh tokens after use and reject a reused token.", 1),
    ("Change the primary key of orders from an integer to a UUID and migrate existing rows.", 1),
    ("Stop using server sessions and issue signed access tokens instead.", 1),
    ("Let organization owners grant and revoke member roles, and persist those grants.", 1),
    ("Record who viewed a customer's credentials in a new audit table.", 1),
    ("Store an encrypted card fingerprint and persist it on the customer.", 1),
    ("Move permission checks out of the API process into a dedicated auth service.", 1),
    ("Send a password-reset link that is a short-lived signed token.", 1),
    ("Add a tenant column and change every query to enforce it.", 1),
    ("Persist the raw OAuth refresh token and change when a new one is issued.", 1),
    ("Drop database-backed sessions and keep login state only in the token.", 1),
    ("Add failed_login_count to users and write the migration.", 1),
    ("Publish a user.created event that a new worker consumes.", 1),
    ("Extract billing out of the API into its own service with its own database.", 1),
    ("Encrypt the notes column for existing customer rows.", 1),
    ("Add an admin impersonation flow and record each impersonation.", 1),
    ("Rotate API keys automatically and invalidate the previous key.", 1),
    ("Change the password hashing algorithm and rehash users on next login.", 1),
    ("Add a refresh_tokens table and stop storing the token on the user row.", 1),
    ("Let a support agent disable another user's sessions immediately.", 1),
    ("Introduce per-workspace billing accounts and migrate old users onto them.", 1),
    ("Replace direct API-to-worker calls with a queue and retry policy.", 1),
    ("Add a break-glass admin role that can read encrypted fields.", 1),
    ("Move the upload service onto a separate network and require service-to-service auth.", 1),
    ("Backfill organization_id on every existing row and start requiring it on write.", 1),
    ("Store webhook signing secrets and verify inbound signatures.", 1),
    ("Change the session cookie so it is not readable by JavaScript, and update the auth middleware to match.", 1),
    ("Add a workflow where deleting a user cascades through the API and the search indexer.", 1),
    ("Split the monolith's notification module into a service that owns its tables.", 1),
    ("Add SAML login for enterprise tenants and persist the identity mapping.", 1),
    ("Rebuild how rate limits are keyed so they are enforced across API replicas.", 1),
    ("Replace the email column with a normalized emails table and migrate existing addresses.", 1),
    ("Allow the mobile app to refresh credentials while offline and reconcile them later.", 1),
    ("Enforce that one user's token cannot call admin routes in every service.", 1),
    ("Migrate secrets from environment files into a managed secret store and change boot to read them there.", 1),
    ("Add end-to-end encryption for attachments so the service cannot see plaintext.", 1),
    ("Rework the invite flow so an invited user gets a role, a tenant, and an audit record.", 1),
    ("Change file ownership from a user to a workspace, including the migration.", 1),
    ("Add a job that re-encrypts stored credentials with a new key.", 1),
]

df = pd.DataFrame(examples, columns=["text", "label"])
df["route"] = df["label"].map(ID2ROUTE)

assert df["text"].is_unique, "Duplicate task text"
counts = df["label"].value_counts()
assert counts[0] == counts[1], counts.to_dict()

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Held out of the labeled split.
paraphrase_df = pd.DataFrame(
    [
        ("Please clean up the docs and correct a spelling mistake.", "luna"),
        ("Users should be able to sign in with Google and their linked identity must be stored.", "astra"),
        ("Add a computed property to User that only exists while the app is running.", "luna"),
        ("Change how refresh tokens are generated, stored, and revoked.", "astra"),
        ("Add one more assertion to the existing test suite.", "luna"),
        ("Correct the spelling of authentication in the developer docs.", "luna"),
        ("Add a comment that describes how refresh tokens are currently revoked.", "luna"),
        ("Show the existing user id in the account header. It is already in the response payload.", "luna"),
        ("Add pagination controls to the admin table using the list the API already returns.", "luna"),
        ("Rename the local variable that holds the migration filename.", "luna"),
        ("The login page says Continue. Change that word to Next. Do not change sign-in.", "luna"),
        ("Add Google login for new users and store their Google subject id.", "astra"),
        ("Change stored user ids from serial integers to UUIDs.", "astra"),
        ("Make session login stateless by issuing JWTs and stop writing the session row.", "astra"),
        ("Add a table that records every time an admin exports customer data.", "astra"),
        ("Split notification sending into a separate service that consumes events.", "astra"),
        ("Persist a new is_admin flag and migrate existing users.", "astra"),
    ],
    columns=["text", "route"],
)

assert not (set(df["text"]) & set(paraphrase_df["text"]))
assert paraphrase_df["text"].is_unique

print("Labeled:", len(df), "Train:", len(train_df), "Test:", len(test_df))
print("Train routes:", train_df["route"].value_counts().to_dict())
print("Paraphrase routes:", paraphrase_df["route"].value_counts().to_dict())
train_df.head()


## 4. Load Gemma 4 E2B in 4-bit

Text-only LoRA. Vision and audio layers stay frozen.


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
)

print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


## 5. Render Gemma 4 conversations

The supervised target is only the route name. `train_on_responses_only` later masks the system and user turns, using Gemma 4 markers:

`<|turn>user` and `<|turn>model`.

Use the non-thinking `gemma-4` template. E2B should answer with the route, not a thought trace.


In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

def as_message(role, text):
    return {
        "role": role,
        "content": [{"type": "text", "text": text}],
    }

def make_conversation(text, route):
    return {
        "conversations": [
            as_message("system", ROUTER_SYSTEM_PROMPT),
            as_message("user", text),
            as_message("assistant", route),
        ]
    }

def to_dataset(frame):
    records = [
        make_conversation(row.text, row.route)
        for row in frame.itertuples(index=False)
    ]
    dataset = Dataset.from_list(records)

    def formatting_prompts_func(examples):
        texts = [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            ).removeprefix("<bos>")
            for convo in examples["conversations"]
        ]
        return {"text": texts}

    return dataset.map(formatting_prompts_func, batched=True)

train_dataset = to_dataset(train_df)

sample = train_dataset[0]["text"]
print(sample)
print()
assert "<|turn>user\n" in sample, "Missing Gemma 4 user turn marker."
assert "<|turn>model\n" in sample, "Missing Gemma 4 model turn marker."
model_span = sample.split("<|turn>model\n", 1)[-1].lstrip()
assert model_span.startswith(("luna", "astra")), model_span[:80]


## 6. Score the base model

Same prompt, before any LoRA update. Later cells reprint this so you can see whether fine-tuning beat the instruction model. Disagreement should show up on the hard negatives, not on "fix a typo" versus "add OAuth".


In [ ]:
import re

ROUTE_RE = re.compile(r"\b(luna|astra)\b", re.IGNORECASE)

def route_task(text, max_new_tokens=8):
    messages = [
        as_message("system", ROUTER_SYSTEM_PROMPT),
        as_message("user", text),
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    raw = tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    ).strip()
    match = ROUTE_RE.search(raw)
    return {
        "route": match.group(1).lower() if match else None,
        "raw": raw,
    }

def score_frame(frame, label):
    rows = []
    for row in frame.itertuples(index=False):
        pred = route_task(row.text)
        rows.append({
            "split": label,
            "text": row.text,
            "gold": row.route,
            "pred": pred["route"],
            "raw": pred["raw"],
            "correct": pred["route"] == row.route,
        })
    scored = pd.DataFrame(rows)
    accuracy = scored["correct"].mean()
    print(f"{label}: {scored['correct'].sum()}/{len(scored)} = {accuracy:.0%}")
    return scored

base_test = score_frame(test_df, "base holdout")
base_paraphrase = score_frame(paraphrase_df, "base paraphrase")
pd.concat([base_test, base_paraphrase], ignore_index=True)[
    ["split", "gold", "pred", "raw", "text"]
]


## 7. Add LoRA adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

model.print_trainable_parameters()


## 8. Train on the route name only

Loss is applied to the `<|turn>model` span, not the task text. The next cell prints that span. It should be the route, not the whole prompt.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir="gemma4-e2b-router-checkpoints",
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        logging_steps=1,
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

row = trainer.train_dataset[0]
trained = tokenizer.decode([x for x in row["labels"] if x != -100])
print("Loss is applied to:\n", repr(trained))
if not any(name in trained.lower() for name in ("luna", "astra")):
    raise RuntimeError(
        "Response masking dropped the route name. "
        "The Gemma 4 turn markers did not match the rendered sample."
    )


## 9. Train


In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Training examples: {len(train_dataset)}")
print(f"Epochs: {EPOCHS}")

trainer_stats = trainer.train()
trainer_stats


## 10. Plot loss

Training loss is logged every optimizer step. Raw per-step loss is noisy; the 5-step moving average shows the trend. The route comparison in the next section is the actual check.


In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame(trainer.state.log_history)
loss_df = (
    loss_df.dropna(subset=["loss"])
    if "loss" in loss_df.columns
    else pd.DataFrame()
)

if len(loss_df):
    loss_df["moving_avg"] = loss_df["loss"].rolling(5).mean()

    plt.figure(figsize=(8, 4))
    plt.plot(loss_df["step"], loss_df["loss"], alpha=0.35, label="Training loss")
    plt.plot(loss_df["step"], loss_df["moving_avg"], linewidth=2, label="5-step average")
    plt.xlabel("Optimizer step")
    plt.ylabel("Loss")
    plt.title("Gemma 4 E2B Router Fine-tuning")
    plt.legend()
    plt.show()
else:
    print("No logged training loss found.")


## 11. Compare base vs fine-tuned routes


In [ ]:
tuned_test = score_frame(test_df, "tuned holdout")
tuned_paraphrase = score_frame(paraphrase_df, "tuned paraphrase")

def accuracy(frame):
    return float(frame["correct"].mean())

comparison = pd.DataFrame(
    [
        {"split": "holdout", "base": accuracy(base_test), "tuned": accuracy(tuned_test)},
        {"split": "paraphrase", "base": accuracy(base_paraphrase), "tuned": accuracy(tuned_paraphrase)},
    ]
)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.0%}"))
tuned = pd.concat([tuned_test, tuned_paraphrase], ignore_index=True)
tuned[["split", "gold", "pred", "correct", "raw", "text"]]


## 12. Route a new task


In [ ]:
route_task(
    "Implement OAuth authentication and add a PostgreSQL migration for refresh tokens."
)


## 13. Save the LoRA adapter

This is the small artifact to keep. Loading it later still requires Gemma 4 E2B.


In [ ]:
OUTPUT_DIR = "gemma4-e2b-model-router-lora"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -qr {OUTPUT_DIR}.zip {OUTPUT_DIR}

try:
    from google.colab import files
    files.download(f"{OUTPUT_DIR}.zip")
except Exception as exc:
    print(f"Saved {OUTPUT_DIR}/ (download skipped: {exc})")


## 14. Optional exports

Merged weights and GGUF use much more disk than the adapter. Uncomment only what you need.


In [ ]:
# MERGED_DIR = "gemma4-e2b-model-router-merged"
# model.save_pretrained_merged(MERGED_DIR, tokenizer)

# GGUF_DIR = "gemma4-e2b-model-router-gguf"
# model.save_pretrained_gguf(
#     GGUF_DIR,
#     tokenizer,
#     quantization_method="Q8_0",
# )

# from huggingface_hub import notebook_login
# notebook_login()
# repo_id = "YOUR_USERNAME/gemma4-e2b-model-router"
# model.push_to_hub(repo_id)
# tokenizer.push_to_hub(repo_id)


## 15. Release GPU memory

Run this last. It drops the model, trainer, and tokenizer, then returns the CUDA cache to the runtime. The LoRA directory and zip on disk are left alone. Route anything else only after loading the adapter again.


In [ ]:
import gc

def reserved_gb():
    if not torch.cuda.is_available():
        return None
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / 1024**3

before = reserved_gb()

for name in ("trainer", "model", "tokenizer"):
    obj = globals().pop(name, None)
    if obj is None:
        continue
    try:
        obj.to("cpu")
    except Exception:
        pass
    del obj

gc.collect()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

after = reserved_gb()
if before is None:
    print("No CUDA device. Dropped model, trainer, and tokenizer.")
else:
    print(f"Reserved GPU memory: {before:.2f} GB -> {after:.2f} GB")


## What this demo does not show

The larger set is still a demo. It is enough to see whether the fine-tune learns a route the base model was not told. It is not enough to trust the route.

For a real router:

1. Collect **500–5,000+** labeled tasks.
2. Keep paraphrases of the same task in the same split.
3. Keep hard negatives: wording about auth or migrations that is actually a bounded change, and the reverse.
4. Compare the fine-tune with the base model on unseen wording before you keep it.
5. Prefer a small encoder when you only need the label. Use this Gemma route when the router itself should be the language model.

### References

- Gemma 4 in Unsloth: https://unsloth.ai/docs/models/gemma-4/train
- Unsloth Gemma 4 E2B: https://huggingface.co/unsloth/gemma-4-E2B-it
- Matching classifier notebook: `colab/ModernBERT_train_classify.ipynb`
